In [ ]:
import numpy as np 
import nibabel as nib 
import matplotlib.pyplot as plt 
import nilearn.plotting as niplt
%matplotlib inline 
import os 
import seaborn as sns 
from scipy.special import softmax 
from scipy.stats import ttest_rel
import pandas as pd 

In [ ]:
# Black-only Figure 2 configuration
BASE = "/nethome/phussein/brains/black_figure2_reproduction"
FIGURE2_SCORES = os.path.join(BASE, "results", "figure2_subject_scores.csv")

story = "black"

print("Using:", FIGURE2_SCORES)
assert os.path.exists(FIGURE2_SCORES), f"Missing: {FIGURE2_SCORES}"


In [ ]:
# excluded sub-296 (<25% comprehension score)
black_subs=['sub-300', 'sub-304', 'sub-293', 'sub-273', 'sub-265', 'sub-307', 'sub-283', 'sub-275',
'sub-291', 'sub-297', 'sub-303', 'sub-294', 'sub-286', 'sub-282', 'sub-310', 'sub-302', 'sub-312',
'sub-301', 'sub-287', 'sub-298', 'sub-313', 'sub-285', 'sub-290', 'sub-288', 'sub-281', 'sub-276', 'sub-277', 'sub-299', 'sub-308',
'sub-272', 'sub-284', 'sub-289', 'sub-280', 'sub-309', 'sub-306', 'sub-127', 'sub-279',
'sub-315', 'sub-314','sub-292', 'sub-311', 'sub-267', 'sub-295', 'sub-305', 'sub-274']

rois=['PostTemp','AntTemp','AngG','IFG','MFG','IFGorb','vmPFC','dmPFC','PMC','HG','V1']
roi_names=['L_'+roi for roi in rois]+['R_'+roi for roi in rois]

assert len(black_subs) == 45


In [ ]:
def get_result_dataset(dataset, rep_name, normalize_isc=True):
    # Minimal Black-only replacement for the original /jukebox loader.
    # The authors' emailed reproduction package already contains the final
    # subject-level percent-noise-ceiling values for Figure 2.
    if dataset != ['black']:
        raise ValueError("This notebook is intentionally Black-only.")

    if not normalize_isc:
        raise ValueError(
            "figure2_subject_scores.csv already contains ISC-normalized scores; "
            "normalize_isc=False is therefore not available here."
        )

    valid_models = {'ling_features', 'glove', 'zrep_magnitudes', 'full', 'full_z'}
    if rep_name not in valid_models:
        raise ValueError(
            f"{rep_name!r} is not contained in the authors' Figure-2 score file. "
            f"Available models: {sorted(valid_models)}"
        )

    scores = pd.read_csv(FIGURE2_SCORES)
    scores = scores[(scores['story'] == 'black') & (scores['model'] == rep_name)]

    # Preserve the original notebook's subjects x 11-ROI shape/order.
    # V1 is not part of Figure 2 and is absent from the authors' score file,
    # so its column remains NaN. The plot uses the other 10 ROIs.
    results = np.full((len(black_subs), len(rois)), np.nan, dtype=float)

    for i, sub in enumerate(black_subs):
        for j, roi in enumerate(rois):
            if roi == 'V1':
                continue

            row = scores[(scores['subject'] == sub) & (scores['roi'] == roi)]
            if len(row) != 1:
                raise ValueError(
                    f"Expected exactly one row for {sub}, {rep_name}, {roi}; found {len(row)}"
                )
            results[i, j] = row.iloc[0]['percent_noise_ceiling']

    return results


In [ ]:
#Figure 2: All representation performances - Black only

attention_scores_performance=get_result_dataset(['black'],'zrep_magnitudes')
ling_features_performance=get_result_dataset(['black'],'ling_features')
glove_performance=get_result_dataset(['black'],'glove')
layer_performance=get_result_dataset(['black'],'full')
z_performance=get_result_dataset(['black'],'full_z')


reps=[ling_features_performance,glove_performance,attention_scores_performance,layer_performance,z_performance]
roi=[]
performance=[]
rep_name=[]
rep_names=['Linguistic Features','GLoVe Embeddings','Transformation Magnitude','Transformer Embeddings','Transformations']
for k in range(len(rep_names)):
    example_layer=reps[k]
    for i in range(example_layer.shape[0]):
        for j in range(11):
            roi.append(rois[j%11])
            rep_name.append(rep_names[k])
            r=example_layer[i,j]
            if r<-100:
                r=-100
            performance.append(r)
roi=np.asarray(roi)
performance=np.asarray(performance)
rep_name=np.asarray(rep_name)
df=pd.DataFrame(dict(performance=performance,roi=roi,rep=rep_name))
plt.figure()
pal=sns.mpl_palette('tab10',n_colors=10)
#pal2=sns.mpl_palette('nipy_spectral',n_colors=len(rep_names))
pal2_cols=['#000000','#A0A0A0','#FFA500','#0000ff','#ff0000','#ff9999']
pal2=sns.color_palette(pal2_cols)

o=['HG','PostTemp','AntTemp','AngG','IFG','IFGorb','MFG','vmPFC','dmPFC','PMC']
#o=['V1']
#v=sns.violinplot(data=df,x='roi',y='performance',zorder=0,inner=None,linewidth=1,cut=0,hue='roi',alpha=0.2)
#sns.stripplot(data=df.sort_values('performance'),x='roi',y='performance',hue='rep',dodge=0.5,zorder=1,size=2,palette=pal2,order=['HG','PostTemp','AntTemp','AngG','IFG',
#'IFGorb','MFG','vmPFC','dmPFC','PMC'],alpha=0.1,hue_order=rep_names)
plt.figure(figsize=(10,5))
sns.pointplot(data=df.sort_values('performance'),x='roi',y='performance',hue='rep',dodge=0.5,join=False,ci=95,n_boot=10000,zorder=2,alpha=0.5,palette=pal2,order=o,capsize=0,hue_order=rep_names,estimator=np.median)
#sns.boxplot(x='roi',y='performance',data=df,saturation=0.35,fliersize=0)
plt.xticks(rotation=90)
#plt.legend(['L','R'],['C0','C1'])
plt.axhline(y=0,color='black')
ax=plt.gca()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:6], labels[:])
sns.despine(top=True,right=True,left=False,bottom=False)
my_colors=['C'+str(i) for i in range(10)]
#for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_colors):
#    ticklabel.set_color(tickcolor)
plt.xlabel("")
plt.ylabel("")
plt.ylim(-5,35)
#for i in range(10):
#    plt.axvline(x=i+0.5,color='black',linestyle='--')
#plt.savefig('plots/fig4_1/v1_right.svg',format='svg')#
#plt.savefig('fig1data_right.svg',format='svg')